# DeepSurv Survival Analysis on METABRIC Dataset

This notebook demonstrates how to apply the DeepSurv model, a deep learning-based extension of the Cox proportional hazards model, to the METABRIC breast cancer dataset. The goal is to predict patient survival times given gene expression and clinical data. We preprocess the data, build and train a neural network survival model, evaluate its performance using C-index and time-dependent AUC, and estimate survival probabilities at clinically relevant time points.

### Importing necessary libraries
We import Python packages for data manipulation (pandas, numpy), machine learning utilities (scikit-learn), deep learning with PyTorch (torchtuples), and survival analysis (pycox). These provide the tools for data preparation, model building, training, and evaluation.

In [30]:
# Load Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torchtuples as tt
import torch
from pycox.models import CoxPH
from pycox.evaluation import EvalSurv

### Loading and preprocessing data
We load the METABRIC dataset, taking care to exclude samples missing important survival endpoint information (overall_survival status and overall_survival_months). This step ensures our model is trained on complete cases with known survival outcomes.

In [3]:
# 1. Load and preprocess dataset

df = pd.read_csv("METABRIC_RNA_Mutation.csv", low_memory=False)

In [4]:
# Remove rows missing overall survival information
surv_cols = ['overall_survival', 'overall_survival_months']
df = df.dropna(subset=surv_cols)

### Defining input features and targets
We separate the survival time and event indicator as the prediction targets and prepare the features by removing identifiers and survival columns. Categorical features are converted to numerical one-hot encodings to be used effectively by the neural network.

In [8]:
# 2. Select features and survival targets

target_duration = df['overall_survival_months'].values.astype('float32')
target_event = df['overall_survival'].values.astype('int64')

# Exclude ID and survival columns from features
X = df.drop(columns=['patient_id', 'overall_survival', 'overall_survival_months'])
X = pd.get_dummies(X, drop_first=True)   # One-hot encode categorical predictors

### Standardising features
Feature scaling is critical for neural network training stability and convergence speed. Here, we standardise each feature to have a zero mean and unit variance using the standard z-score normalisation.

In [10]:
# 3. Standardize features

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Splitting data into training, validation, and testing
The dataset is randomly split into training (60%) used to fit the model, validation (20%) for tuning and early stopping, and testing (20%) for unbiased performance evaluation. A fixed random seed ensures reproducibility.

In [12]:
# 4. Train/val/test split

x_train, x_temp, dur_train, dur_temp, ev_train, ev_temp = train_test_split(
    X_scaled, target_duration, target_event, test_size=0.4, random_state=42
)
x_val, x_test, dur_val, dur_test, ev_val, ev_test = train_test_split(
    x_temp, dur_temp, ev_temp, test_size=0.5, random_state=42
)

### Ensuring correct data types
PyTorch requires input arrays to be in specific formats. Therefore, all feature arrays are converted to float32 and event indicators to int64. This enforces compatibility and avoids subtle bugs.

In [14]:
# Ensure correct data types
# Fix: force float32 for features, durations
x_train = np.asarray(x_train, dtype='float32')
x_val   = np.asarray(x_val, dtype='float32')
x_test  = np.asarray(x_test, dtype='float32')

dur_train = np.asarray(dur_train, dtype='float32')
dur_val   = np.asarray(dur_val, dtype='float32')
dur_test  = np.asarray(dur_test, dtype='float32')

ev_train = np.asarray(ev_train, dtype='int64')
ev_val   = np.asarray(ev_val, dtype='int64')
ev_test  = np.asarray(ev_test, dtype='int64')

y_train = (dur_train, ev_train)
y_val   = (dur_val, ev_val)
y_test  = (dur_test, ev_test)

### Creating the DeepSurv neural network architecture
We build a multilayer perceptron with three hidden layers: 128, 64, and 32 nodes respectively. Batch normalization and dropout are incorporated to reduce overfitting and improve generalization. This architecture maps patient features into risk scores for survival prediction.

In [16]:
# 5. Build DeepSurv neural network

in_features = x_train.shape[1]
num_nodes = [128, 64, 32]
net = tt.practical.MLPVanilla(
    in_features, num_nodes, 1, batch_norm=True, dropout=0.1, output_bias=False
)

### Training the DeepSurv model
The model is trained with the Adam optimiser, using a learning rate of 0.01. Training proceeds in mini-batches of 256 samples, for a maximum of 512 epochs. Early stopping with a patience of 20 epochs monitors validation performance and halts training when improvements cease, helping avoid overfitting.

In [18]:
# 6. Train DeepSurv model
model = CoxPH(net, tt.optim.Adam(0.01))
batch_size = 256
epochs = 512
callbacks = [tt.callbacks.EarlyStopping(patience=20)]

log = model.fit(
    x_train, y_train, batch_size, epochs,
    callbacks, val_data=(x_val, y_val), verbose=True
)

0:	[0s / 0s],	
1:	[0s / 0s],	
2:	[0s / 0s],	
3:	[0s / 0s],	
4:	[0s / 0s],	
5:	[0s / 0s],	
6:	[0s / 0s],	
7:	[0s / 0s],	
8:	[0s / 0s],	
9:	[0s / 0s],	
10:	[0s / 0s],	
11:	[0s / 0s],	
12:	[0s / 0s],	
13:	[0s / 0s],	
14:	[0s / 0s],	
15:	[0s / 0s],	
16:	[0s / 0s],	
17:	[0s / 0s],	
18:	[0s / 0s],	
19:	[0s / 0s],	


### Evaluating model performance on the test set
We compute the baseline hazard function and predict survival curves for each test sample. The concordance index (C-index), quantifying the model’s ability to correctly rank patient risks, is calculated. The C-index is also saved for integration with other analyses.

In [20]:
# 7. Evaluate metrics on test set

model.compute_baseline_hazards()
surv = model.predict_surv_df(x_test)

ev = EvalSurv(surv, dur_test, ev_test, censor_surv='km')
c_index = ev.concordance_td()

print(f"DeepSurv C-index: {c_index:.4f}")

# Save C-index to txt file (for upstream integration)
with open("outputs/DeepSurv_cindex.txt", "w") as f:
    f.write(f"DeepSurv C-index: {c_index:.4f}\n")

DeepSurv C-index: 0.2409


### Calculating time-dependent AUC scores
In addition to the C-index, we measure model discrimination over time by computing the integrated Brier score (interpreted here as time-dependent AUC) at 12, 36, and 60 months. This metric assesses accuracy of survival predictions at multiple critical time points.

In [22]:
# 8. Time-dependent AUC at 12, 36, 60 months

eval_times = [12, 36, 60]
auc_dict = {t: ev.integrated_brier_score(np.array([t])) for t in eval_times}
print("Time-dependent AUCs:", auc_dict)

# Save as CSV
pd.DataFrame(list(auc_dict.items()), columns=["Time_in_Months", "AUC"]).to_csv(
    "outputs/DeepSurv_AUC.csv", index=False
)

Time-dependent AUCs: {12: nan, 36: nan, 60: nan}


/opt/anaconda3/lib/python3.12/site-packages/pycox/evaluation/ipcw.py:87: RuntimeWarning: invalid value encountered in scalar divide
  return integral / (time_grid[-1] - time_grid[0])


### Estimating survival probabilities at 5 and 10 years
We aggregate predicted survival curves over all test patients and extract mean survival probabilities at clinically relevant 5- and 10-year time points. These probabilities provide interpretable survival likelihood estimates for the cohort and are saved for reporting.

In [24]:
# 9. Mean survival probabilities at 5 and 10 years

# Predict survival for each test sample
surv_preds = surv.T  # transpose: samples x time

# Survival times in months
time_points = surv_preds.columns.values
idx5  = np.argmin(np.abs(time_points - 60))   # 5 years
idx10 = np.argmin(np.abs(time_points - 120))  # 10 years

surv5  = surv_preds.iloc[:, idx5].mean()
surv10 = surv_preds.iloc[:, idx10].mean()

sr = pd.DataFrame({"Model": ["DeepSurv"], "Surv5y": [surv5], "Surv10y": [surv10]})

# Save as CSV for summary comparison
sr.to_csv("outputs/DeepSurv_survival_rates.csv", index=False)
print(sr)

print("DeepSurv results saved in outputs/")

      Model    Surv5y   Surv10y
0  DeepSurv  0.957462  0.756281
DeepSurv results saved in outputs/
